# 03 - Real World: IT-Grundschutz PDF -> JSON -> Qdrant -> RAG

In diesem Notebook bauen wir eine vollständige, praxisnahe Ingestion- und Retrieval-Pipeline auf Basis eines echten Dokuments:
- `notebooks/raw_data/standard_200_1.pdf`

Warum dieser Ablauf?
- **Docling** extrahiert strukturierte Inhalte aus komplexen PDFs.
- **Normalisierung** behebt typische OCR-/Encoding-Artefakte (z. B. `/C231` statt `ü`).
- **Chunking** macht Inhalte für Embeddings und semantische Suche verarbeitbar.
- **Metadaten** sorgen für Nachvollziehbarkeit der Treffer (Quelle, Strategie, Parameter).
- **Seitennummern aus JSON (`prov.page_no`)** ermöglichen belastbare Zitate und bessere Quellenangaben.

Ziel: Am Ende führen wir eine RAG-Query gegen eine Qdrant-Collection aus.


## 1) Konfiguration (Pfade, Collection, Chunk-Parameter)

Warum diese Parameter wichtig sind:
- `MAX_CHUNK`: Maximale Zeichenlänge pro Chunk (Kontextfenster/Embedding-Qualität).
- `OVERLAP`: Überlappung zwischen Chunks zur Verringerung von Kontextverlust an Chunk-Grenzen.
- `COLLECTION_NAME`: Ziel-Collection in Qdrant für diesen Versuch.
- `OPENAI_API_KEY`: wird für LiteLLM-Embeddings benötigt (aus `notebooks/.env`).


In [ ]:
from pathlib import Path
import json
import os
import re
from typing import List

REPO_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
NOTEBOOKS_DIR = REPO_ROOT / 'notebooks'
RAW_DIR = NOTEBOOKS_DIR / 'raw_data'
OUT_DIR = NOTEBOOKS_DIR / 'processed'
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Lade .env aus notebooks/ (falls vorhanden)
env_path = NOTEBOOKS_DIR / '.env'
if env_path.exists():
    for line in env_path.read_text(encoding='utf-8').splitlines():
        line = line.strip()
        if not line or line.startswith('#') or '=' not in line:
            continue
        k, v = line.split('=', 1)
        os.environ.setdefault(k.strip(), v.strip())

PDF_PATH = RAW_DIR / 'standard_200_1.pdf'

COLLECTION_NAME = 'it_grundschutz_docling'
QDRANT_HOST = 'localhost'
QDRANT_PORT = 6333

# LiteLLM/OpenAI Embedding Modell
EMBED_MODEL_NAME = 'openai/octen-embedding-8b'
LLM_MODEL_NAME = 'openai/gpt-oss-120b'
API_BASE_URL = os.getenv('OPENAI_API_BASE', 'https://api.aisc.hpi.de/')

MAX_CHUNK = 1200
OVERLAP = 200

print('PDF:', PDF_PATH)
print('PDF exists:', PDF_PATH.exists())
print('OUT_DIR:', OUT_DIR)
print('Collection:', COLLECTION_NAME)
print('Embedding model:', EMBED_MODEL_NAME)
print('LLM model:', LLM_MODEL_NAME)
print('API base URL:', API_BASE_URL)
print('OPENAI_API_KEY set:', bool(os.getenv('OPENAI_API_KEY')))
print('MAX_CHUNK:', MAX_CHUNK, '| OVERLAP:', OVERLAP)


In [ ]:
if not PDF_PATH.exists():
    raise FileNotFoundError(f'PDF not found: {PDF_PATH}')
print('PDF found. Ready for Docling conversion.')


## 2) Text-Normalisierung und Chunking-Bausteine

Hier definieren wir die Kernlogik zur Qualitätsverbesserung der extrahierten Texte:
- Korrektur von Umlaut-/Encoding-Artefakten (`/C196`, `/C231`, ...)
- JSON-Normalisierung nur für inhaltliche Textfelder (`text`, `orig`)
- Parser-orientiertes Chunking entlang semantischer Blöcke (Absatz-basiert)


In [ ]:
_UMLAUT_MAP = {
    "C196": "Ä",
    "C214": "Ö",
    "C218": "Ö",
    "C220": "Ü",
    "C216": "Ä",
    "C219": "Ü",
    "C228": "ä",
    "C229": "ä",
    "C230": "ö",
    "C231": "ü",
    "C246": "ö",
    "C252": "ü",
    "C223": "ß",
}

def _fix_german_umlauts(text: str) -> str:
    """Ersetzt Docling/OCR-Platzhalter wie /C231 durch echte Umlaute."""
    def repl(match: re.Match) -> str:
        code = match.group(1)
        if not code.startswith("C"):
            code = f"C{code}"
        return _UMLAUT_MAP.get(code, match.group(0))

    # Erfasst sowohl /C231 als auch C231
    text = re.sub(r"/?(C\d{3})", repl, text)
    text = re.sub(r"[ \t]{2,}", " ", text)
    return text

def count_umlaut_placeholders(obj) -> int:
    s = json.dumps(obj, ensure_ascii=False)
    return len(re.findall(r"/?C\d{3}", s))

def normalize_json(doc_json: dict) -> dict:
    """Normalisiert nur inhaltliche Textfelder, nicht Metadaten."""
    def walk(obj):
        if isinstance(obj, dict):
            out = {}
            for k, v in obj.items():
                if k in {"text", "orig"} and isinstance(v, str):
                    out[k] = _fix_german_umlauts(v)
                else:
                    out[k] = walk(v)
            return out
        if isinstance(obj, list):
            return [walk(x) for x in obj]
        return obj
    return walk(doc_json)

def normalize_text(text: str) -> str:
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = _fix_german_umlauts(text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

def parser_aware_split(markdown_text: str, max_chunk: int = 1200, overlap: int = 200) -> List[str]:
    """Chunking entlang Absätzen; zu lange Blöcke werden mit Overlap geteilt."""
    if overlap >= max_chunk:
        raise ValueError("overlap must be smaller than max_chunk")

    text = normalize_text(markdown_text)
    if not text:
        return []

    blocks = text.split("\n\n")
    chunks = []
    current = ""

    for block in blocks:
        candidate = (current + "\n\n" + block).strip() if current else block
        if len(candidate) <= max_chunk:
            current = candidate
            continue

        if current:
            chunks.append(current)

        if len(block) <= max_chunk:
            current = block
        else:
            start = 0
            while start < len(block):
                end = min(start + max_chunk, len(block))
                chunks.append(block[start:end])
                if end == len(block):
                    break
                start = end - overlap
            current = ""

    if current:
        chunks.append(current)

    if overlap > 0 and chunks:
        with_overlap = [chunks[0]]
        for i in range(1, len(chunks)):
            prefix = chunks[i - 1][-overlap:]
            with_overlap.append((prefix + "\n" + chunks[i]).strip())
        return with_overlap

    return chunks


## 3) Docling-Konvertierung: PDF -> Markdown + JSON

Wir erzeugen zwei Artefakte:
- **Markdown**: gut lesbar und ideal für header-/absatzbasiertes Chunking
- **JSON**: strukturreich (Provenance, Seitenbezug, Bounding Boxes) für tiefergehende Analysen


In [ ]:
from docling.document_converter import DocumentConverter

def docling_pdf_to_markdown_and_json(pdf_path: Path):
    converter = DocumentConverter()
    result = converter.convert(str(pdf_path))
    doc = result.document

    # Markdown für chunking/retrieval optimieren
    markdown = normalize_text(doc.export_to_markdown())

    # Strukturierte JSON-Repräsentation laden und Textfelder normalisieren
    doc_json = doc.export_to_dict()
    before = count_umlaut_placeholders(doc_json)
    doc_json_clean = normalize_json(doc_json)
    after = count_umlaut_placeholders(doc_json_clean)

    print(f"Umlaut-Placeholder ersetzt: {before - after}")
    print(f"Verbleibende Placeholder: {after}")
    return markdown, doc_json_clean

markdown_text, docling_json = docling_pdf_to_markdown_and_json(PDF_PATH)

markdown_out = OUT_DIR / f"{PDF_PATH.stem}.md"
json_out = OUT_DIR / f"{PDF_PATH.stem}.docling.json"
markdown_out.write_text(markdown_text, encoding="utf-8")
json_out.write_text(json.dumps(docling_json, ensure_ascii=False, indent=2), encoding="utf-8")

print("Saved markdown:", markdown_out)
print("Saved json:", json_out)
print("Markdown chars:", len(markdown_text))


## 4) Chunking-Strategien

Wir definieren drei Strategien, um Retrieval-Qualität vergleichbar zu machen:
- `markdown_headers`: Splitting entlang Markdown-Überschriften, dann Feinsplitting bei Bedarf
- `json_text_no_chunk`: keine künstliche Chunk-Grenze, ein Record pro JSON-`text` Feld
- `json_structured_sections`: strukturgetrieben aus Docling-JSON (`content_layer`, `label`, `prov.page_no`)

Wichtig für `json_structured_sections`:
- `content_layer == "furniture"` (Header/Footer/Seitennummern) wird vorab entfernt.
- Abschnittslabels (`section_heading`, `page_header`, ...) sind die Chunk-Grenzen.
- **Kein `MAX_CHUNK`, kein `OVERLAP`**: ein Chunk läuft exakt von Heading zu Heading.
- Seiteninformationen bleiben als Metadaten erhalten und verbessern Zitation/Quellenangaben.


In [ ]:
import re
from typing import List, Dict, Any

def chunk_markdown_by_headers(markdown_text: str, max_chunk: int = 1200, overlap: int = 200) -> List[str]:
    """Splitte entlang Markdown-Überschriften; große Abschnitte werden nachgeteilt."""
    text = normalize_text(markdown_text)
    if not text:
        return []

    parts = re.split(r'(?m)^(#{1,6}\s.+)$', text)
    sections = []
    if parts and parts[0].strip():
        sections.append(parts[0].strip())

    for i in range(1, len(parts), 2):
        header = parts[i].strip()
        body = parts[i + 1].strip() if i + 1 < len(parts) else ""
        sections.append(f"{header}\n\n{body}".strip())

    chunks: List[str] = []
    for sec in sections:
        if len(sec) <= max_chunk:
            chunks.append(sec)
        else:
            chunks.extend(parser_aware_split(sec, max_chunk=max_chunk, overlap=overlap))

    return chunks

def records_from_markdown_header_chunks(markdown_text: str) -> List[Dict[str, Any]]:
    chunks = chunk_markdown_by_headers(markdown_text, max_chunk=MAX_CHUNK, overlap=OVERLAP)
    return [
        {
            "chunk_id": i,
            "text": chunk_text,
            "metadata": {
                "source_file": PDF_PATH.name,
                "source_path": str(PDF_PATH),
                "doc_type": "pdf",
                "converter": "docling",
                "chunking_mode": "markdown_headers",
                "max_chunk": MAX_CHUNK,
                "overlap": OVERLAP,
                "total_chunks": len(chunks),
                # Aus purem Markdown sind exakte Seitenzahlen i. d. R. nicht zuverlässig ableitbar
                "page_numbers": [],
                "citation_hint": None,
            },
        }
        for i, chunk_text in enumerate(chunks)
    ]


In [ ]:
def records_from_docling_json_text_fields(doc_json: dict) -> List[Dict[str, Any]]:
    """Ein Record pro JSON-Textfeld, inkl. Seitenzahlen aus prov.page_no."""
    items = []

    def extract_page_numbers(node: dict):
        pages = []
        prov = node.get("prov", [])
        if isinstance(prov, list):
            for p in prov:
                if isinstance(p, dict) and isinstance(p.get("page_no"), int):
                    pages.append(p["page_no"])
        return sorted(set(pages))

    def walk(obj):
        if isinstance(obj, dict):
            text_value = obj.get("text")
            if isinstance(text_value, str) and text_value.strip():
                pages = extract_page_numbers(obj)
                citation_hint = f"S. {', '.join(map(str, pages))}" if pages else None
                items.append({
                    "text": normalize_text(text_value),
                    "page_numbers": pages,
                    "citation_hint": citation_hint,
                })
            for v in obj.values():
                walk(v)
        elif isinstance(obj, list):
            for x in obj:
                walk(x)

    walk(doc_json)

    # Kein zusätzliches Chunking/Overlap: 1 Record = 1 extrahiertes Textsegment
    return [
        {
            "chunk_id": i,
            "text": item["text"],
            "metadata": {
                "source_file": PDF_PATH.name,
                "source_path": str(PDF_PATH),
                "doc_type": "pdf",
                "converter": "docling",
                "chunking_mode": "json_text_no_chunk",
                "max_chunk": None,
                "overlap": 0,
                "total_chunks": len(items),
                "page_numbers": item["page_numbers"],
                "citation_hint": item["citation_hint"],
            },
        }
        for i, item in enumerate(items)
    ]


## 4b) Strukturgetriebenes Chunking aus Docling-JSON

Diese Variante nutzt die semantischen Signale aus Docling direkt:
- `content_layer: furniture` wird verworfen
- Labels wie `section_heading` starten neue semantische Abschnitte
- `prov.page_no` wird als Seitenmetadatum pro Chunk gespeichert

Wichtig: In dieser Variante wird **nicht** weiter nach Länge gesplittet.
Ein Chunk entspricht immer genau einem Abschnitt von Heading zu Heading.


In [ ]:
def records_from_docling_json_structured_sections(doc_json: dict) -> List[Dict[str, Any]]:
    """Strukturgetriebene Chunks: exakt von Heading zu Heading, ohne Max-Chunk/Overlap."""
    elements = doc_json.get('texts', [])
    if not isinstance(elements, list):
        return []

    # Labels, die als neue Abschnittsgrenze dienen
    heading_labels = {'section_header', 'section_heading', 'heading', 'title'}

    def extract_page_numbers(node: dict) -> List[int]:
        pages = []
        prov = node.get('prov', [])
        if isinstance(prov, list):
            for p in prov:
                if isinstance(p, dict) and isinstance(p.get('page_no'), int):
                    pages.append(p['page_no'])
        return sorted(set(pages))

    def format_citation(pages: List[int]):
        if not pages:
            return None
        if len(pages) == 1:
            return f'S. {pages[0]}'
        return f'S. {pages[0]}-{pages[-1]}'

    items = []
    current_heading = ''
    current_parts: List[str] = []
    current_pages: List[int] = []

    def flush_section():
        nonlocal current_parts, current_pages
        if not current_parts:
            return

        body = normalize_text(' '.join(current_parts))
        if not body:
            current_parts = []
            current_pages = []
            return

        section_text = f'{current_heading}\n\n{body}'.strip() if current_heading else body
        page_numbers = sorted(set(current_pages))

        items.append({
            'text': section_text,
            'page_numbers': page_numbers,
            'citation_hint': format_citation(page_numbers),
        })

        current_parts = []
        current_pages = []

    for el in elements:
        if not isinstance(el, dict):
            continue

        # Furniture (Header/Footer/Seitennummern) ausschließen
        if el.get('content_layer') == 'furniture':
            continue

        raw_text = el.get('text') or el.get('orig') or ''
        if not isinstance(raw_text, str) or not raw_text.strip():
            continue

        text = normalize_text(raw_text)
        if not text:
            continue

        label = (el.get('label') or '').strip()
        pages = extract_page_numbers(el)

        if label in heading_labels or (label == 'page_header' and el.get('content_layer') != 'furniture'):
            flush_section()
            current_heading = text
            current_pages.extend(pages)
            continue

        current_parts.append(text)
        current_pages.extend(pages)

    flush_section()

    return [
        {
            'chunk_id': i,
            'text': item['text'],
            'metadata': {
                'source_file': PDF_PATH.name,
                'source_path': str(PDF_PATH),
                'doc_type': 'pdf',
                'converter': 'docling',
                'chunking_mode': 'json_structured_sections',
                'max_chunk': None,
                'overlap': 0,
                'total_chunks': len(items),
                'page_numbers': item['page_numbers'],
                'citation_hint': item['citation_hint'],
            },
        }
        for i, item in enumerate(items)
    ]


## 4c) Strategie wählen und `records` erzeugen

Hier wird festgelegt, welche Chunking-Strategie tatsächlich in Qdrant geladen wird.
Der gewählte Modus wird in den Metadaten gespeichert (`chunking_mode`).


In [ ]:
# Optionen: 'markdown_headers' | 'json_text_no_chunk' | 'json_structured_sections'
CHUNKING_MODE = 'json_structured_sections'

if CHUNKING_MODE == 'markdown_headers':
    records = records_from_markdown_header_chunks(markdown_text)
elif CHUNKING_MODE == 'json_text_no_chunk':
    records = records_from_docling_json_text_fields(docling_json)
elif CHUNKING_MODE == 'json_structured_sections':
    records = records_from_docling_json_structured_sections(docling_json)
else:
    raise ValueError(f'Unknown CHUNKING_MODE: {CHUNKING_MODE}')

chunks_out = OUT_DIR / f'{PDF_PATH.stem}.{CHUNKING_MODE}.chunks.json'
chunks_out.write_text(json.dumps(records, ensure_ascii=False, indent=2), encoding='utf-8')

print('Chunking mode:', CHUNKING_MODE)
print('Anzahl Records:', len(records))
print('Saved chunks:', chunks_out)
if records:
    print('\nPreview record 0:\n')
    print(records[0]['text'][:800])
    print('Pages:', records[0]['metadata'].get('page_numbers'))
    print('Citation:', records[0]['metadata'].get('citation_hint'))


## 5) Embeddings mit LiteLLM + Upload in Qdrant

Hier verwenden wir LiteLLM als einheitliche API für Embeddings.
Vorteil: Modell-/Provider-Wechsel mit minimalen Codeänderungen.

Wichtig: Der Upload erfolgt in Batches, damit Qdrant-Limits für Request-Größe nicht überschritten werden.


In [ ]:
from litellm import embedding
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

if not os.getenv('OPENAI_API_KEY'):
    raise ValueError('OPENAI_API_KEY fehlt. Bitte in notebooks/.env setzen.')

def _extract_embedding(item):
    if isinstance(item, dict):
        return item.get('embedding')
    return getattr(item, 'embedding', None)

def embed_texts_litellm(texts, model, batch_size=64):
    vectors = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        resp = embedding(model=model, input=batch, api_base=API_BASE_URL, encoding_format='float')
        for item in resp.data:
            vec = _extract_embedding(item)
            if vec is None:
                raise RuntimeError('Embedding response ohne Vektor erhalten.')
            vectors.append(vec)
    return vectors

def batched(iterable, batch_size):
    for i in range(0, len(iterable), batch_size):
        yield iterable[i:i+batch_size]

client = QdrantClient(host=QDRANT_HOST, port=QDRANT_PORT)

texts = [r['text'] for r in records]
vectors = embed_texts_litellm(texts, model=EMBED_MODEL_NAME)
vector_size = len(vectors[0])

# Neu statt deprecated recreate_collection
if client.collection_exists(COLLECTION_NAME):
    client.delete_collection(COLLECTION_NAME)
client.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=VectorParams(size=vector_size, distance=Distance.COSINE),
)

points = []
for r, vec in zip(records, vectors):
    payload = {
        'text': r['text'],
        **r['metadata'],
        'chunk_id': r['chunk_id'],
    }
    points.append(PointStruct(id=r['chunk_id'], vector=vec, payload=payload))

# Batch-Upload, um Request-Limit (32MB) nicht zu reißen
UPSERT_BATCH_SIZE = 64
for batch_idx, batch_points in enumerate(batched(points, UPSERT_BATCH_SIZE), start=1):
    client.upsert(collection_name=COLLECTION_NAME, points=batch_points)
    print(f'Uploaded batch {batch_idx} ({len(batch_points)} points)')

info = client.get_collection(COLLECTION_NAME)
print('Collection ready:', COLLECTION_NAME)
print('Vectors count:', info.points_count)
print('Vector size:', vector_size)
print('Total points uploaded:', len(points))


## 6) RAG Query Demo

Wir suchen semantisch ähnliche Chunks zur Nutzerfrage und zeigen die Top-Treffer mit Score.


In [ ]:
def rag_search(query: str, top_k: int = 5):
    q_vec = embed_texts_litellm([query], model=EMBED_MODEL_NAME)[0]
    response = client.query_points(
        collection_name=COLLECTION_NAME,
        query=q_vec,
        limit=top_k,
        with_payload=True,
        with_vectors=False,
    )

    hits = response.points
    results = []
    for h in hits:
        payload = h.payload or {}
        results.append({
            'score': h.score,
            'chunk_id': payload.get('chunk_id'),
            'text': payload.get('text', ''),
            'source_file': payload.get('source_file'),
            'page_numbers': payload.get('page_numbers', []),
            'citation_hint': payload.get('citation_hint'),
        })
    return results

query = 'Welche Anforderungen stellt der Standard an Informationssicherheit und Risikomanagement?'
hits = rag_search(query, top_k=5)

print('QUERY:', query)
print('=' * 100)
for i, h in enumerate(hits, start=1):
    pages = h['page_numbers'] if h['page_numbers'] else '-'
    cite = h['citation_hint'] if h['citation_hint'] else '-'
    print(f'[{i}] score={h["score"]:.4f} | chunk_id={h["chunk_id"]} | source={h["source_file"]} | pages={pages} | cite={cite}')
    print(h['text'])
    print('-' * 100)


## 7) RAG + LLM Antwort mit LiteLLM

Hier wird aus den Top-Retrieval-Treffern ein Kontext gebaut und an ein Chat-LLM gesendet.
Standardmodell: `openai/gpt-oss-120b` (über euren LiteLLM-Gateway).
Falls euer Deployment einen anderen Namen hat, setze `LLM_MODEL_NAME` entsprechend in `.env`.


In [ ]:
from litellm import completion
from IPython.display import Markdown, display


def build_rag_context(hits, max_chars_per_chunk: int = 1400) -> str:
    blocks = []
    for i, h in enumerate(hits, start=1):
        pages = h.get('page_numbers') or []
        cite = h.get('citation_hint') or '-'
        pages_text = ', '.join(map(str, pages)) if pages else '-'
        snippet = (h.get('text') or '')[:max_chars_per_chunk]
        blocks.append(
            f"[Quelle {i}] score={h['score']:.4f} | pages={pages_text} | cite={cite}\n{snippet}"
        )
    return '\n\n'.join(blocks)

def answer_with_llm(query: str, hits, model: str = None) -> str:
    model = model or LLM_MODEL_NAME
    context = build_rag_context(hits)

    system_prompt = (
        'Du bist ein RAG-Assistent für IT-Grundschutz. '
        'Beantworte nur auf Basis des bereitgestellten Kontexts. '
        'Wenn die Information fehlt, sage das klar. '
        'Gib am Ende Quellen mit Seitennummern an.'
    )

    user_prompt = f"Frage:\n{query}\n\nKontext:\n{context}"

    resp = completion(
        model=model,
        messages=[
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': user_prompt},
        ],
        api_base=API_BASE_URL,
        api_key=os.getenv('OPENAI_API_KEY'),
        temperature=0.2,
    )

    return resp.choices[0].message.content

llm_answer = answer_with_llm(query, hits)
print('LLM answer:\n')
display(Markdown(llm_answer))


## 8) Optional: Mini-Chat (Retrieval-only)

Kein LLM nötig: Es werden pro Frage die relevantesten Chunks angezeigt.
Das ist hilfreich, um Retrieval-Qualität und Chunking-Strategie schnell zu vergleichen.


In [ ]:
from IPython.display import Markdown, display

def rag_chat(turns: int = 3, top_k: int = 5):
    print("Mini-Chat gestartet. Tippe 'exit' zum Beenden.")
    for _ in range(turns):
        q = input("\nFrage: ").strip()
        if q.lower() in {"exit", "quit"}:
            print("Chat beendet.")
            break

        hits = rag_search(q, top_k=top_k)
        answer = answer_with_llm(q, hits, model=LLM_MODEL_NAME)

        # Antwort als Markdown rendern
        display(Markdown(f"### Antwort\n\n{answer}"))

        # Quellen kompakt anzeigen
        print("\nQuellen:")
        for i, h in enumerate(hits, start=1):
            pages = h.get("page_numbers") or "-"
            cite = h.get("citation_hint") or "-"
            print(f"[{i}] score={h['score']:.4f} | pages={pages} | cite={cite}")

# Start
rag_chat(turns=10, top_k=5)


## 9) Ergebnis und Einordnung

Du hast jetzt eine vollständige Pipeline umgesetzt:
1. PDF mit Docling extrahiert
2. Encoding-Artefakte bereinigt
3. Chunking-Strategie gewählt
4. Embeddings + Metadaten in Qdrant gespeichert
5. RAG-Abfrage durchgeführt
6. LLM-Antwort auf Basis der gefundenen Quellen erzeugt

Empfehlung für den Workshop:
- `markdown_headers`: gut für schnelle, robuste Baselines auf Markdown.
- `json_text_no_chunk`: nah an der Roh-Extraktion, inkl. Seitenbezug je Textfeld.
- `json_structured_sections`: klare Abschnittsgrenzen (Heading->Heading), ohne künstliches Splitting.
- Vergleiche Trefferqualität und Antworttreue mit identischer Query über alle Strategien.
